In [1]:
%load_ext autoreload
%autoreload 2

import os
ROOT_PATH = os.path.join(os.getcwd(), os.pardir)
import sys
sys.path.append(ROOT_PATH) 
import pandas as pd
import numpy as np
import notebooks.utils as notebooks_utils
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm


/tmp/ipykernel_3525955/1646661068.py:8: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Using tag <-> tlabel matching

In [ ]:
fins, labels = notebooks_utils.load_data(after_year=None)

Initially, there are 258 positives
After None, there are 258 positives


In [53]:
tag_tlabel_mapping = pd.read_csv('/home/andreasideras/research_ideas/BankruptcyPredictionAD/tag_tlabel_mapping.csv')
tag_tlabel_mapping.shape[0]

692768

In [82]:
tag_tlabel_mapping.tag.unique().__len__()

496794

In [83]:
tag_tlabel_mapping.tlabel.unique().__len__()

640468

In [74]:
fins_tags = set(fins.columns)
tag_tlabel_mapping_subset = tag_tlabel_mapping.loc[tag_tlabel_mapping.tag.isin(fins_tags)]
tag_tlabel_mapping_subset = tag_tlabel_mapping_subset[['tag', 'tlabel']]


In [75]:
tag_tlabel_mapping_subset

,tag,tlabel
0,AccountsAndNotesReceivableNet,"Accounts and Financing Receivable, after Allow..."
1,AccountsAndOtherReceivablesNetCurrent,"Accounts and Other Receivables, Net, Current"
2,AccountsNotesAndLoansReceivableNetCurrent,"Accounts and Financing Receivable, after Allow..."
3,AccountsPayableAndAccruedLiabilitiesCurrent,"Accounts Payable and Accrued Liabilities, Current"
4,AccountsPayableAndAccruedLiabilitiesCurrentAnd...,Accounts Payable and Accrued Liabilities
...,...,...
691448,PreferredStockLiquidationPreferenceValue,"Preferred stock, Liquidation value"
691991,WeightedAverageNumberOfShareOutstandingBasicAn...,Weighted average number of common shares outst...
692355,ContributionsFromNoncontrollingInterests,Contributions from noncontrolling interest
692364,WeightedAverageNumberOfShareOutstandingBasicAn...,Weighted average number of shares outstanding ...


In [78]:
multiple_labels = tag_tlabel_mapping_subset.groupby('tag').count() > 1
multiple_labels[multiple_labels.tlabel].head(10)

,tlabel
tag,
AccountsAndOtherReceivablesNetCurrent,True
AccountsPayable,True
AccountsPayableAndAccruedLiabilities,True
AccountsPayableAndAccruedLiabilitiesCurrent,True
AccountsPayableAndOtherAccruedLiabilities,True
AccountsPayableAndOtherAccruedLiabilitiesCurrent,True
AccountsPayableCurrent,True
AccountsPayableInterestBearingCurrent,True
AccountsPayableInterestBearingNoncurrent,True


In [81]:
tag_tlabel_mapping[tag_tlabel_mapping.tag == 'AccountsPayableAndAccruedLiabilities']

,Unnamed: 0,tag,tlabel
8639,11239,AccountsPayableAndAccruedLiabilities,Accounts payable and accrued liabilities
47394,57244,AccountsPayableAndAccruedLiabilities,Accounts Payable and Accrued Liabilities
73751,91999,AccountsPayableAndAccruedLiabilities,Accounts Payable And Accrued Liabilities
124444,96614,AccountsPayableAndAccruedLiabilities,AccountsPayableAndAccruedLiabilities
155036,88450,AccountsPayableAndAccruedLiabilities,[Accounts payable and accrued liabilities]
238169,3,AccountsPayableAndAccruedLiabilities,Accounts Payable and Accrued Liabilities (Depr...
490959,97438,AccountsPayableAndAccruedLiabilities,Deficit attributable to shareholders
509805,11014,AccountsPayableAndAccruedLiabilities,Accounts payable and accrued liabilities (Note 4)
514815,76667,AccountsPayableAndAccruedLiabilities,Accounts payable and accrued expenses
639050,88844,AccountsPayableAndAccruedLiabilities,[Deficit attributable to shareholders]


In [64]:
df = tag_tlabel_mapping[['tag', 'tlabel']]
df['tlabel'] = df['tlabel'].apply(lambda x: str(x).lower())
tag_to_label = dict(zip(df['tag'], df['tlabel']))
fins_cols = list(fins.columns)
df_translated = fins.copy().rename(columns=tag_to_label)

In [50]:
df

,tag,tlabel
0,AccountsAndNotesReceivableNet,"accounts and financing receivable, after allow..."
1,AccountsAndOtherReceivablesNetCurrent,"accounts and other receivables, net, current"
2,AccountsNotesAndLoansReceivableNetCurrent,"accounts and financing receivable, after allow..."
3,AccountsPayableAndAccruedLiabilitiesCurrent,"accounts payable and accrued liabilities, current"
4,AccountsPayableAndAccruedLiabilitiesCurrentAnd...,accounts payable and accrued liabilities
...,...,...
692763,StockIssuedDuringPeriodValueExerciseOfStockPur...,stock issued during period value exercise of s...
692764,StockIssuedDuringPeriodValueExerciseOfStockRig...,stock issued during period value exercise of s...
692765,StockIssuedDuringPeriodValueSharesIssuedForExt...,stock issued during period value shares issued...
692766,StockIssuedDuringPeriodValueSharesIssuedForExt...,stock issued during period value shares issued...


In [ ]:
for idx, row in tqdm(fins.iterrows()):
    for tag in row.index:
        if tag in tag_tlabel_mapping:
            df_translated.at[idx,tag_to_label[tag]] = row[tag]
df_translated.index = fins.index
    

56974it [09:20, 101.60it/s]


Slicing using tlabel mapping

In [52]:
df_translated.notna().sum().sum() - fins.notna().sum().sum()

0

Slicing

In [ ]:
# fins, labels = notebooks_utils.load_data(after_year=None)
perc=0.28
sliced_index, sliced_columns = notebooks_utils.get_sliced_perc(fins, labels, cols_perc=0.27, false_rows_perc=perc, true_rows_perc=perc)
print(notebooks_utils.test_predictors(fins.loc[sliced_index,sliced_columns], labels.loc[sliced_index], n_splits=10))
print()
print("Tags density:")
(fins.loc[sliced_index,sliced_columns].notna().sum(axis=0)/(fins.loc[sliced_index,sliced_columns].shape[0])).sort_values(ascending=False).head(20)

Select only columns that have more than 27.0% dense rows on the original dataset
Then:
Select only false rows that have more than 28.000000000000004% dense columns 
Select only true rows that have more than 28.000000000000004% dense columns 

There are 107 columns

Mean sample density (out of 107 feats):
68.55

Std sample density (out of 107 feats):
15.52

label
False    33558
True       185
Name: count, dtype: int64

Baseline - Logistic Regression:
                         Mean       Std
F1 Score             0.043803  0.005518
Average Precision    0.064040  0.018858
Recall at Precision  0.097661  0.062960

Tags density:


Assets                                             0.989509
LiabilitiesAndStockholdersEquity                   0.984234
IncomeTaxExpenseBenefit                            0.967430
CashAndCashEquivalentsAtCarryingValue              0.936283
StockholdersEquity                                 0.926681
RetainedEarningsAccumulatedDeficit                 0.926118
NetIncomeLoss                                      0.918027
PropertyPlantAndEquipmentNet                       0.884806
WeightedAverageNumberOfSharesOutstandingBasic      0.882613
NetCashProvidedByUsedInInvestingActivities         0.881931
NetCashProvidedByUsedInFinancingActivities         0.880509
DeferredIncomeTaxExpenseBenefit                    0.879175
NetCashProvidedByUsedInOperatingActivities         0.878019
EarningsPerShareBasic                              0.877930
WeightedAverageNumberOfDilutedSharesOutstanding    0.875352
EarningsPerShareDiluted                            0.870077
AccumulatedOtherComprehensiveIncomeLossN

In [ ]:
fins2010, labels2010 = notebooks_utils.load_data(after_year=2010)
sliced_index, sliced_columns = notebooks_utils.get_sliced_perc(fins2010, labels2010, cols_perc=0.70, false_rows_perc=1, true_rows_perc=0.85)
notebooks_utils.test_predictors(fins2010.loc[sliced_index,sliced_columns], labels2010.loc[sliced_index], num_runs=10)

Select only columns that have more than 70.0% dense rows on the original dataset
Then:
Select only false rows that have more than 100% dense columns 
Select only true rows that have more than 85.0% dense columns 

There are 33 columns

label
False    2275
True       56
Name: count, dtype: int64

Baseline - Logistic Regression:


,Mean,Std
F1 Score,0.293565,0.050495
Average Precision,0.358417,0.182019
Recall at Precision,0.336667,0.199694


In [ ]:
fins, labels = notebooks_utils.load_data(after_year=None)
sliced_index, sliced_columns = notebooks_utils.get_sliced_perc(fins, labels, cols_perc=0.453, false_rows_perc=1, true_rows_perc=1)
notebooks_utils.test_predictors(fins.loc[sliced_index,sliced_columns], labels.loc[sliced_index], num_runs=10)

Select only columns that have more than 45.300000000000004% dense rows on the original dataset
Then:
Select only false rows that have more than 100% dense columns 
Select only true rows that have more than 100% dense columns 

There are 30 columns

label
False    1921
True        7
Name: count, dtype: int64


In [3]:
new_fins,new_labels = notebooks_utils.maximize_positive_density(fins, labels, top_cols=15)
notebooks_utils.test_predictors(new_fins, new_labels, num_runs=5)

Index(['label', 'Assets', 'LiabilitiesAndStockholdersEquity',
       'CashAndCashEquivalentsAtCarryingValue', 'IncomeTaxExpenseBenefit',
       'RetainedEarningsAccumulatedDeficit', 'PropertyPlantAndEquipmentNet',
       'LiabilitiesCurrent', 'AssetsCurrent', 'NetIncomeLoss',
       'StockholdersEquity', 'OperatingIncomeLoss', 'ShareBasedCompensation',
       'DeferredIncomeTaxExpenseBenefit',
       'CommonStockParOrStatedValuePerShare', 'CommonStockSharesAuthorized'],
      dtype='object')
label
False    11580
True        69
Name: count, dtype: int64

Baseline - Logistic Regression:


,Mean,Std
F1 Score,0.064462,0.005758
Average Precision,0.129416,0.080041
Recall at Precision,0.142857,0.127775


Fill with data from finsim

In [89]:
sec2finsim, sec_filled = notebooks_utils.expand_with_finsim(fins,match_percentage=0.001,top=20)
new_fins,new_labels = notebooks_utils.maximize_positive_density(sec_filled, labels, top_cols=15)
notebooks_utils.test_predictors(new_fins, new_labels, num_runs=5)

516054 new values from finsim!
Index(['label', 'CashAndCashEquivalentsAtCarryingValue', 'Assets',
       'LiabilitiesAndStockholdersEquity', 'IncomeTaxExpenseBenefit',
       'PropertyPlantAndEquipmentNet', 'RetainedEarningsAccumulatedDeficit',
       'NetIncomeLoss', 'LiabilitiesCurrent', 'AssetsCurrent',
       'OperatingIncomeLoss', 'StockholdersEquity',
       'DeferredIncomeTaxExpenseBenefit', 'ShareBasedCompensation',
       'NetCashProvidedByUsedInInvestingActivities',
       'NetCashProvidedByUsedInOperatingActivities'],
      dtype='object')
label
False    16328
True        75
Name: count, dtype: int64

Baseline - Logistic Regression:


,Mean,Std
F1 Score,0.048565,0.003854
Average Precision,0.092753,0.032147
Recall at Precision,0.106667,0.090431


In [60]:
sec2finsim2010, sec_filled2010 = notebooks_utils.expand_with_finsim(fins2010,match_percentage=0.001,top=20)
sliced_index, sliced_columns = notebooks_utils.get_sliced_perc(sec_filled2010, labels2010, cols_perc=0.85, false_rows_perc=1, true_rows_perc=1)
notebooks_utils.test_predictors(sec_filled2010.loc[sliced_index,sliced_columns], labels2010.loc[sliced_index], num_runs=10)

487871 new values from finsim!
Select only columns that have more than 85.0% dense rows on the original dataset
Then:
Select only false rows that have more than 100% dense columns 
Select only true rows that have more than 100% dense columns 

There are 15 columns

label
False    17834
True        68
Name: count, dtype: int64

Baseline - Logistic Regression:


,Mean,Std
F1 Score,0.050890,0.010333
Average Precision,0.079354,0.037146
Recall at Precision,0.102381,0.092857


Future Internet paper data

In [70]:
fi = pd.read_csv('/home/andreasideras/research_ideas/BankruptcyPredictionAD/data/future_internet_paper/future_internet_fins.csv')
fi.set_index('uid', inplace=True)
print(fi.label.value_counts())
print(fi.label.sum()/fi.shape[0])
notebooks_utils.test_predictors(fi.iloc[:,:-1], fi.iloc[:,-1], num_runs=10)

label
False    13425
True       196
Name: count, dtype: int64
0.014389545554658248

Baseline - Logistic Regression:


,Mean,Std
F1 Score,0.107830,0.005965
Average Precision,0.164516,0.047717
Recall at Precision,0.173421,0.094396
